In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import numpy as np
import pandas as pd

### Loading pickle file and model 

In [3]:
# Loading the trained model
model = load_model('model.h5')
#loading the scaler
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f) 

#loading the lablencoder for gender coloum
with open('label_encoder_gender.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)

#loading the one-hotencoder for geography coloum
with open('one_hot_encoder_geography.pkl','rb') as f:
    label_encoder_geo=pickle.load(f)

### Loading the Data for making prediction using out model

In [4]:
data=pd.read_csv('Dataset/Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
data.columns.to_list()

['RowNumber',
 'CustomerId',
 'Surname',
 'CreditScore',
 'Geography',
 'Gender',
 'Age',
 'Tenure',
 'Balance',
 'NumOfProducts',
 'HasCrCard',
 'IsActiveMember',
 'EstimatedSalary',
 'Exited']

In [6]:
data=data.drop(['RowNumber','CustomerId','Surname','Exited'],axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,619,France,Female,42,2,0.00,1,1,1,101348.88
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58
2,502,France,Female,42,8,159660.80,3,1,0,113931.57
3,699,France,Female,39,1,0.00,2,0,0,93826.63
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10


### Making the inputr Data Suitable for prediction

In [7]:
data['Gender']=label_encoder_gender.transform(data['Gender'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,619,France,0,42,2,0.00,1,1,1,101348.88
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58
2,502,France,0,42,8,159660.80,3,1,0,113931.57
3,699,France,0,39,1,0.00,2,0,0,93826.63
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10


In [8]:
geography_data=label_encoder_geo.transform(data[['Geography']])
geography_df=pd.DataFrame(geography_data,columns=label_encoder_geo.get_feature_names_out(['Geography']))
data=data.drop(['Geography'],axis=1)
data=pd.concat([data,geography_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0.0,1.0


### Scaling the data

In [9]:
data=scaler.transform(data)

In [10]:
data

array([[-0.33880827, -1.09499335,  0.29493847, ...,  0.01595384,
        -0.57946723, -0.57638802],
       [-0.45296331, -1.09499335,  0.19986603, ...,  0.21057501,
        -0.57946723,  1.73494238],
       [-1.5530028 , -1.09499335,  0.29493847, ...,  0.23472492,
        -0.57946723, -0.57638802],
       ...,
       [ 0.59518752, -1.09499335, -0.27549615, ..., -1.01443762,
        -0.57946723, -0.57638802],
       [ 1.24898458,  0.91324755,  0.29493847, ..., -0.13114365,
         1.72572313, -0.57638802],
       [ 1.4565392 , -1.09499335, -1.03607564, ..., -1.08215522,
        -0.57946723, -0.57638802]], shape=(10000, 11))

## Making the prediction using the trained model

In [11]:
#prediction output
prediction=model.predict(data)
prediction

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 541us/step


array([[0.23134756],
       [0.14829217],
       [0.16083837],
       ...,
       [0.15759794],
       [0.04484176],
       [0.3245024 ]], shape=(10000, 1), dtype=float32)

In [12]:
prediction = model.predict(data)
pred = (prediction >= 0.5).astype(int).flatten()

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 477us/step


In [13]:
actual_answers=pd.read_csv('Dataset/Churn_Modelling.csv')['Exited']
actual_answers.head()

0    1
1    0
2    1
3    0
4    0
Name: Exited, dtype: int64

## finding the accuracy of model on actual Data

In [14]:
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
print("Accuracy:", accuracy_score(actual_answers, pred))
print("Confusion Matrix:\n", confusion_matrix(actual_answers, pred))
print("Classification Report:\n", classification_report(actual_answers, pred))

Accuracy: 0.7419
Confusion Matrix:
 [[7265  698]
 [1883  154]]
Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.91      0.85      7963
           1       0.18      0.08      0.11      2037

    accuracy                           0.74     10000
   macro avg       0.49      0.49      0.48     10000
weighted avg       0.67      0.74      0.70     10000

